# Optional experiment: householdâ€“coupon relevance

The main project remains audience selection for one hypothetical USD 2 offer.
This independent notebook ranks real historical coupon identifiers for known
TypeB/TypeC campaign recipients. It is an offline recommendation experiment,
not an uplift model, a live campaign or an experiment with random assignment.

We ask whether eligible products are bought during the evaluation window.
Such purchases can happen without a coupon; this is relevance, not impact.

## 1. Load the validated purchase source and historical offer catalogue

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/retail_customer_360/metrics.py").is_file()
)
sys.path.insert(0, str(ROOT / "src"))
from retail_customer_360.metrics import load_transactions

con = load_transactions(ROOT)
OUT = ROOT / "data/processed/coupon_recommendations"
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
for table, filename in {
    "campaigns": "campaign_desc.csv",
    "contacts": "campaign_table.csv",
    "mapping_raw": "coupon.csv",
    "redemptions": "coupon_redempt.csv",
}.items():
    con.execute(
        f"CREATE TEMP TABLE {table} AS SELECT * FROM read_csv_auto(?)",
        [str(ROOT / "data/raw" / filename)],
    )
observation_end = con.execute("SELECT max(day) FROM raw_transactions").fetchone()[0]
display(pd.DataFrame({"last_observed_day": [observation_end]}))

,last_observed_day
0,711


## 2. Define a historical decision point

Each decision is made immediately before campaign START_DAY. Features use only
positive-quantity, positive-sales purchases in the preceding 180 days.
Require START_DAY > 180 and the entire campaign to end within observation.
Evaluate from START_DAY through min(END_DAY, START_DAY + 29), inclusive.
This can be shorter than 30 days; duration is reported per campaign.
Start/end inclusivity is an explicit working convention.

Only recorded recipients of TypeB/TypeC are eligible: the source describes their
full campaign coupon set. Exclude TypeA because individual allocations are unknown.
No claim is made that a recipient saw an offer or that its full terms are known.
The catalogue is treated as available at campaign start; creation timestamps are absent.

In [3]:
con.execute("""
CREATE TEMP TABLE decisions AS
SELECT campaign, description, start_day, end_day,
       least(end_day, start_day + 29) AS outcome_end
FROM campaigns
WHERE description IN ('TypeB', 'TypeC')
  AND start_day > 180 AND end_day <= ?
""", [observation_end])
con.execute("""
CREATE TEMP TABLE mapping AS
SELECT DISTINCT campaign, coupon_upc, product_id FROM mapping_raw
""")
display(con.execute("SELECT * FROM decisions ORDER BY start_day").df())

,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY,outcome_end
0,27,TypeC,237,300,266
1,28,TypeB,259,320,288
2,29,TypeB,281,334,310
3,1,TypeB,346,383,375
4,2,TypeB,351,383,380
5,3,TypeC,356,412,385
6,4,TypeB,372,404,401
7,5,TypeB,377,411,406
8,6,TypeC,393,425,422
9,7,TypeB,398,432,427


In [4]:
catalogue_audit = con.execute("""
SELECT (SELECT count(*) FROM mapping_raw) AS raw_mapping_rows,
       (SELECT count(*) FROM mapping) AS unique_mapping_rows,
       (SELECT count(*) FROM decisions) AS evaluated_campaigns
""").df()
display(catalogue_audit)
assert len(con.execute("SELECT * FROM decisions").df()) > 0
con.execute("""
CREATE TEMP TABLE paid AS
SELECT DISTINCT household_key, basket_id, day, product_id
FROM raw_transactions WHERE quantity > 0 AND sales_value > 0
""")

,raw_mapping_rows,unique_mapping_rows,evaluated_campaigns
0,124548,119384,24


## 3. Generate candidates and explainable scores

One candidate is a household/campaign/coupon triple. Keep recipients with at
least one paid purchase in the prior 180 days; report this restricted population.
Every mapped campaign coupon is a candidate, including zero-affinity offers.

Personal score: number of distinct past baskets containing an eligible product.
Break ties by most recent eligible-product purchase, then historical population
popularity, then coupon identifier. Baseline: population basket count alone.
Population counts use all households, with the same strictly historical window.
Count baskets once per coupon even when several eligible products were bought.
This simple ranking favours familiar products and broad offers; it does not
identify households persuaded by a discount or optimise profit.

In [5]:
con.execute("""
CREATE TEMP TABLE eligible AS
SELECT DISTINCT c.household_key, d.*
FROM contacts c JOIN decisions d USING (campaign)
WHERE EXISTS (
    SELECT 1 FROM paid p WHERE p.household_key = c.household_key
      AND p.day >= d.start_day - 180 AND p.day < d.start_day
)
""")
con.execute("""
CREATE TEMP TABLE history AS
SELECT d.campaign, p.* FROM decisions d JOIN paid p
ON p.day >= d.start_day - 180 AND p.day < d.start_day
""")

In [6]:
con.execute("""
CREATE TEMP TABLE affinity AS
SELECT h.campaign, h.household_key, m.coupon_upc,
       count(DISTINCT h.basket_id) AS personal_baskets,
       max(h.day) AS last_product_day
FROM history h JOIN mapping m USING (campaign, product_id)
GROUP BY h.campaign, h.household_key, m.coupon_upc
""")
con.execute("""
CREATE TEMP TABLE popularity AS
SELECT h.campaign, m.coupon_upc,
       count(DISTINCT (h.household_key, h.basket_id)) AS popular_baskets
FROM history h JOIN mapping m USING (campaign, product_id)
GROUP BY h.campaign, m.coupon_upc
""")

In [7]:
con.execute("""
CREATE TEMP TABLE candidates AS
SELECT e.*, m.coupon_upc,
       coalesce(a.personal_baskets, 0) AS personal_baskets,
       coalesce(a.last_product_day, -1) AS last_product_day,
       coalesce(p.popular_baskets, 0) AS popular_baskets
FROM eligible e
JOIN (SELECT DISTINCT campaign, coupon_upc FROM mapping) m USING (campaign)
LEFT JOIN affinity a USING (campaign, household_key, coupon_upc)
LEFT JOIN popularity p USING (campaign, coupon_upc)
""")

In [8]:
con.execute("""
CREATE TEMP TABLE ranked AS
SELECT *, row_number() OVER (
    PARTITION BY household_key, campaign
    ORDER BY personal_baskets DESC, last_product_day DESC,
             popular_baskets DESC, coupon_upc
) AS personal_rank,
row_number() OVER (
    PARTITION BY household_key, campaign
    ORDER BY popular_baskets DESC, coupon_upc
) AS baseline_rank
FROM candidates
""")

In [9]:
coverage = con.execute("""
SELECT d.campaign, d.start_day, d.outcome_end - d.start_day + 1 AS days,
       count(DISTINCT c.household_key) AS recorded_recipients,
       count(DISTINCT e.household_key) AS eligible_recipients
FROM decisions d LEFT JOIN contacts c USING (campaign)
LEFT JOIN eligible e
ON e.campaign = c.campaign AND e.household_key = c.household_key
GROUP BY d.campaign, d.start_day, d.outcome_end ORDER BY d.start_day
""").df()
display(coverage)

,CAMPAIGN,START_DAY,days,recorded_recipients,eligible_recipients
0,27,237,30,12,12
1,28,259,30,17,17
2,29,281,30,118,118
3,1,346,30,13,13
4,2,351,30,48,48
5,3,356,30,12,12
6,4,372,30,81,81
7,5,377,30,166,166
8,6,393,30,65,65
9,7,398,30,198,198


## 4. Reveal future outcomes only after ranking

A positive relevance label means at least one eligible product was purchased
in a paid basket during the evaluation window. A redemption label is reported
separately and uses the same window; it is not inferred from product purchases.
Both labels use EXISTS, avoiding inflation from many-to-many coupon mappings.
Overlapping coupons can receive a positive label from the same purchase.

In [10]:
con.execute("""
CREATE TEMP TABLE evaluated AS
SELECT r.*, EXISTS (
    SELECT 1 FROM paid p JOIN mapping m USING (product_id)
    WHERE m.campaign = r.campaign AND m.coupon_upc = r.coupon_upc
      AND p.household_key = r.household_key
      AND p.day BETWEEN r.start_day AND r.outcome_end
)::INTEGER AS product_hit,
EXISTS (
    SELECT 1 FROM redemptions x
    WHERE x.campaign = r.campaign AND x.coupon_upc = r.coupon_upc
      AND x.household_key = r.household_key
      AND x.day BETWEEN r.start_day AND r.outcome_end
)::INTEGER AS redeemed
FROM ranked r
""")

## 5. Compare top-three recommendations with popularity

Precision@3 is the fraction of selected offers with a positive product label.
Hit rate@3 is the fraction of household/campaign decisions with at least one hit.
Use min(3, available offers) if fewer than three candidates exist.
Average decisions equally; households appearing in several campaigns repeat.
Redemption precision is a descriptive secondary metric, not a response probability.
Rules and K=3 are fixed in this notebook, not tuned against these outcomes.

In [11]:
con.execute("""
CREATE TEMP TABLE selected AS
SELECT *, 'personal_history' AS method FROM evaluated WHERE personal_rank <= 3
UNION ALL
SELECT *, 'population_popularity' AS method FROM evaluated
WHERE baseline_rank <= 3
""")
per_decision = con.execute("""
SELECT method, campaign, household_key, count(*) AS recommended,
       avg(product_hit) AS precision_at_3, max(product_hit) AS hit_at_3,
       avg(redeemed) AS redemption_precision_at_3
FROM selected GROUP BY method, campaign, household_key
""").df()
per_decision.columns = per_decision.columns.str.lower()

In [12]:
metrics = ["precision_at_3", "hit_at_3", "redemption_precision_at_3"]
summary = per_decision.groupby("method")[metrics].mean()
summary["decisions"] = per_decision.groupby("method").size()
by_campaign = per_decision.groupby(["campaign", "method"])[metrics].mean()
display(summary.round(4))
display(by_campaign.round(4))

,precision_at_3,hit_at_3,redemption_precision_at_3,decisions
method,,,,
personal_history,0.1918,0.4052,0.0203,3129
population_popularity,0.1364,0.3247,0.0154,3129


precision_at_3  hit_at_3  \
campaign method                                            
1        personal_history               0.1282    0.3846   
         population_popularity          0.1282    0.3077   
2        personal_history               0.1458    0.3542   
         population_popularity          0.0972    0.2708   
3        personal_history               0.1111    0.2500   
         population_popularity          0.0833    0.1667   
4        personal_history               0.1564    0.3951   
         population_popularity          0.0658    0.1975   
5        personal_history               0.0904    0.2048   
         population_popularity          0.0803    0.1867   
6        personal_history               0.0308    0.0308   
         population_popularity          0.0308    0.0308   
7        personal_history               0.0471    0.1313   
         population_popularity          0.0404    0.1212   
9        personal_history               0.2367    0.5000   
         population_popularity          0.1667    0.3750   
10       personal_history               0.1057    0.2439   
         population_popularity          0.0786    0.2114   
11       personal_history               0.0748    0.1869   
         population_popularity          0.0639    0.1729   
12       personal_history               0.3549    0.6941   
         population_popularity          0.3176    0.6824   
14       personal_history               0.2158    0.4598   
         population_popularity          0.1562    0.4018   
15       personal_history               0.2059    0.2941   
         population_popularity          0.2059    0.2941   
16       personal_history               0.1135    0.2660   
         population_popularity          0.0887    0.2287   
17       personal_history               0.2112    0.4752   
         population_popularity          0.1007    0.2673   
19       personal_history               0.2205    0.4615   
         population_popularity          0.1718    0.3846   
20       personal_history               0.1844    0.3689   
         population_popularity          0.1311    0.3197   
21       personal_history               0.1077    0.3077   
         population_popularity          0.0564    0.1385   
22       personal_history               0.2645    0.5290   
         population_popularity          0.1775    0.4022   
23       personal_history               0.2969    0.5847   
         population_popularity          0.1876    0.4372   
25       personal_history               0.3476    0.6791   
         population_popularity          0.2478    0.5722   
27       personal_history               0.0833    0.2500   
         population_popularity          0.0556    0.1667   
28       personal_history               0.2549    0.5882   
         population_popularity          0.2353    0.5882   
29       personal_history               0.2203    0.4746   
         population_popularity          0.1384    0.3390   

                                redemption_precision_at_3  
campaign method                                            
1        personal_history                          0.0256  
         population_popularity                     0.0000  
2        personal_history                          0.0278  
         population_popularity                     0.0208  
3        personal_history                          0.0278  
         population_popularity                     0.0278  
4        personal_history                          0.0123  
         population_popularity                     0.0082  
5        personal_history                          0.0120  
         population_popularity                     0.0100  
6        personal_history                          0.0154  
         population_popularity                     0.0154  
7        personal_history                          0.0067  
         population_popularity                     0.0051  
9        personal_history                          0.0246  
         po

## 6. Sanity checks and local exports

These checks verify cutoff boundaries, unique candidates and complete windows.
Recipient availability is retrospective campaign metadata, not random assignment.
Exports include household identifiers and stay in Git-ignored data/processed.
Only aggregate results are displayed here. No main pipeline or USD 2 policy changes.

In [13]:
assert con.execute("""
SELECT count(*) FROM history h JOIN decisions d USING (campaign)
WHERE h.day >= d.start_day OR h.day < d.start_day - 180
""").fetchone()[0] == 0
assert con.execute("""
SELECT count(*) FROM (
    SELECT household_key, campaign, coupon_upc FROM ranked
    GROUP BY ALL HAVING count(*) > 1
)
""").fetchone()[0] == 0
last_outcome = con.execute("SELECT max(outcome_end) FROM decisions").fetchone()[0]
assert last_outcome <= observation_end
assert per_decision.recommended.between(1, 3).all()

In [14]:
recommendations = con.execute("""
SELECT household_key, campaign, coupon_upc, start_day, outcome_end,
       personal_rank, personal_baskets, last_product_day, popular_baskets
FROM ranked WHERE personal_rank <= 3
ORDER BY campaign, household_key, personal_rank
""").df()
recommendations.to_csv(OUT / "recommendations.csv", index=False)
summary.to_csv(OUT / "evaluation_summary.csv")
by_campaign.to_csv(OUT / "evaluation_by_campaign.csv")
coverage.to_csv(OUT / "population_coverage.csv", index=False)
display(pd.DataFrame({"exported_recommendations": [len(recommendations)]}))
con.close()

,exported_recommendations
0,9240


## Observed offline results

Evaluated 24 historical campaigns. Figures below are descriptive;
they are not incremental effects or evidence of profitable targeting.

- personal_history: product precision@3 19.2%; hit rate@3 40.5%; redemption precision@3 2.0%; 3,129 household/campaign decisions.
- population_popularity: product precision@3 13.6%; hit rate@3 32.5%; redemption precision@3 1.5%; 3,129 household/campaign decisions.

Personal history performs better on these relevance measures if its values
exceed the popularity baseline. This may reward predictable repeat purchases,
precisely the purchases a business may not want to subsidise. Ranking relevance
alone cannot resolve that tension; see the causal limitations below.

## 7. What we can and cannot conclude

- A higher product hit rate supports relevance on these historical recipients;
  it does not show incremental sales, prevented churn or savings in marketing cost.
  Read the per-campaign results before generalising an aggregate difference.
- There is no untreated counterfactual. Purchases happened under historical
  marketing, including all received offers, not only our proposed top three.
- TypeA is excluded; inactive recipients, new households and campaigns without
  180 days of prior observation are outside this evaluation population.
- Campaign starts are separate historical cutoffs. Later campaigns may use
  earlier observed purchases legitimately; repeated households and overlapping
  windows mean the decisions are not independent statistical observations.
- Coupon face value, minimum spend, detailed conditions, funding and product
  margin are absent. Dates and product mappings are necessary, not sufficient,
  checks for operational eligibility. These coupons are historical, not live.
- Exact-product history has cold-start limits. Broad coupons have more chances
  to match a purchase. Several coupons may cover the same products; no diversity,
  redemption stacking or stock constraint is modelled.
- There is no claimed online A/B experiment, causal uplift estimate or profit
  optimisation. Main-project USD 2 scenario assumptions do not become the
  actual value of these coupon identifiers.

### Next step: a prospective test, not performed here

Validate offer terms and costs first. Randomise eligible households into no-offer,
popular-offer and personalised-offer groups, with comparable offer economics.
Keep assignment at household level and prevent overlapping experimental contacts.
Predefine the observation window, sample-size calculation, primary incremental
contribution metric and guardrails. Compare intention-to-treat outcomes, including
discount/contact costs and purchases without redemption. Until this is available,
use the notebook as a recommendation prototype rather than a spending policy.